# Actividad 1

## Primer lectura de la colección Time

Con la colección "Time", consistente de 423 documentos con 83 consultas asociadas con sus correspondientes juicios de relevancia, hacer lo que se pide en el readme

### Formato de la colección

TIME.ALL es un solo archivo con los 423 documentos concatenados... 
En todo el file hay tags de *TEXT y *STOP
Un doc va desde su TEXT inicial hasta el siguiente y el final del file es el STOP

In [114]:
from pathlib import Path
import string
from IPython.display import display


RUTA_COLECCION = Path("../time")
ARCHIVO_DOCUMENTOS = RUTA_COLECCION / "TIME.ALL"

# MARKERS
MARCA_DOCUMENTO = "*TEXT"
MARCA_FIN = "*STOP"


In [55]:
def parse_hdr(linea):
    """
    Extrae los campos de una linea *TEXT

    Inputs:
    -------
    linea: Linea completa que empieza con el marcador *TEXT

    Returns:
    -------
    dict: Llaves id, fecha y pagina, las tres como cadena

    """
    campos = linea.split()
    return {
        "id": campos[1],
        "fecha": "/".join(campos[2:-2]),
        "pagina": campos[-1],
    }

In [56]:
def leer_documentos(ruta):
    """
    Separa TIME.ALL en sus documentos.
    Recorre el archivo una sola vez acumulando lineas hasta topar con
    el siguiente marcador

    Inputs:
    -------
    ruta: Ruta al archivo TIME.ALL

    Returns:
    -------
    list: Un dict por documento, con id, fecha, pagina y texto

    """
    docs = []
    actual = None
    cuerpo = []

    for line in ruta.read_text(encoding="ascii").splitlines():
        line = line.strip()

        if line.startswith(MARCA_DOCUMENTO):
            if actual is not None:
                actual["texto"] = " ".join(cuerpo)
                docs.append(actual)
            actual = parse_hdr(line)
            cuerpo = []
        elif line == MARCA_FIN:
            break
        elif line:
            cuerpo.append(line)

    # El *STOP cierra el ultimo documento, pero si el archivo terminara
    # sin el, el que quedo abierto se guarda de todas formas
    if actual is not None:
        actual["texto"] = " ".join(cuerpo)
        docs.append(actual)

    return docs

In [59]:
docs = leer_documentos(ARCHIVO_DOCUMENTOS)

print(f"documentos recuperados: {len(docs)}")

print("\n\n--- SANITY CHECK ---\n")
print(f"id: {docs[0]['id']}")
print(f"fecha: {docs[0]['fecha']}")
print(f"pagina: {docs[0]['pagina']}")
print(f"largo: {len(docs[0]['texto'])} caracteres")
print()
print(f"texto original: {docs[0]['texto'][:300]} ...")

documentos recuperados: 423


--- SANITY CHECK ---

id: 017
fecha: 01/04/63
pagina: 020
largo: 5649 caracteres

texto original: THE ALLIES AFTER NASSAU IN DECEMBER 1960, THE U.S . FIRST PROPOSED TO HELP NATO DEVELOP ITS OWN NUCLEAR STRIKE FORCE . BUT EUROPE MADE NO ATTEMPT TO DEVISE A PLAN . LAST WEEK, AS THEY STUDIED THE NASSAU ACCORD BETWEEN PRESIDENT KENNEDY AND PRIME MINISTER MACMILLAN, EUROPEANS SAW EMERGING THE FIRST O ...


### Hasta aqui tenemos los docs recuperados... pero los organizamos en un diccionario/tabla hash pq los vamos a recorrer muchas veces y asi, para que sea optimo. Ahora procesaremos los datos

## Eliminacion de palabras vacias

In [5]:
RUTA_COLECCION = Path("../time")
ARCHIVO_NULL_W = RUTA_COLECCION / "TIME.STP"
NULL_WORDS = set()

with open(ARCHIVO_NULL_W) as f:
    NULL_WORDS = {linea.strip().lower() for linea in f if linea.strip()}

print(f"palabras nulas cargadas: {len(NULL_WORDS)}")

palabras nulas cargadas: 340


In [6]:
def limpiar_token(token):
    """
    Quita laos signos de puntuacion pegada a los extremos de una 
    palabra

    Inputs:
    -------
    token: Palabra tal como salio de partir el texto por espacios

    Returns:
    -------
    str: El token sin signos de punt

    """
    return token.strip(string.punctuation)


def eliminar_palabras(doc_txt):
    """
    Elimina las palabras vacias del texto del documento

    Inputs:
    -------
    doc_txt: Un string que representa el texto del documento

    Returns:
    -------
    str: El mismo texto, sin palabras vacias ni puntuacion suelta

    """
    limpios = (limpiar_token(palabra) for palabra in doc_txt.split())
    return " ".join(
        palabra
        for palabra in limpios
        if palabra and palabra.lower() not in NULL_WORDS
    )


In [61]:
# guardams en otro atributa ara comparar despues
vc = 0
for doc in docs:
    doc["texto_nn"] = eliminar_palabras(doc["texto"])
    vc += 1

print(f"documentos procesados (eliminar palabras nulas): {vc}")


print("\n\n--- SANITY CHECK ---\n")
print(f"id: {docs[0]['id']}")
print(f"fecha: {docs[0]['fecha']}")
print(f"pagina: {docs[0]['pagina']}")
print(f"largo: {len(docs[0]['texto'])} caracteres")
print(f"texto original: {len(docs[0]['texto'].split())} palabras")
print(f"texto sin nulos: {len(docs[0]['texto_nn'].split())} palabras")
print()
print(f"texto original: {docs[0]['texto'][:300]} ...")
print(f"texto sin nulos: {docs[0]['texto_nn'][:300]} ...")


documentos procesados (eliminar palabras nulas): 423


--- SANITY CHECK ---

id: 017
fecha: 01/04/63
pagina: 020
largo: 5649 caracteres
texto original: 980 palabras
texto sin nulos: 445 palabras

texto original: THE ALLIES AFTER NASSAU IN DECEMBER 1960, THE U.S . FIRST PROPOSED TO HELP NATO DEVELOP ITS OWN NUCLEAR STRIKE FORCE . BUT EUROPE MADE NO ATTEMPT TO DEVISE A PLAN . LAST WEEK, AS THEY STUDIED THE NASSAU ACCORD BETWEEN PRESIDENT KENNEDY AND PRIME MINISTER MACMILLAN, EUROPEANS SAW EMERGING THE FIRST O ...
texto sin nulos: ALLIES NASSAU DECEMBER 1960 U.S PROPOSED HELP NATO DEVELOP NUCLEAR STRIKE FORCE EUROPE ATTEMPT DEVISE PLAN STUDIED NASSAU ACCORD PRESIDENT KENNEDY PRIME MINISTER MACMILLAN EUROPEANS SAW EMERGING OUTLINES NUCLEAR NATO U.S WANTS SUPPORT SPRANG ANGLO-U.S CRISIS CANCELLATION BUG-RIDDEN SKYBOLT MISSILE U ...


## Truncamiento con el algoritmo de Porter

El algoritmo de Porter es basicamente un truncador de palabras en ingles, que no usa diccinario ni es un lematizaor.
se basa en la medida m. Cada palabra se ve como C(VC)^mV, donde C es una racha de consonantes y V de vocales. m cuenta cuantas veces se repite el par VC, o sea qué tan larga es la raiz... convierte palabras asi 

- studies      -> studi
- happy        -> happi
- allies       -> alli
- university   -> univers


In [63]:
from nltk.stem import PorterStemmer

# algoritmo original
PORTER_ORIG = PorterStemmer(mode=PorterStemmer.ORIGINAL_ALGORITHM)

# nltk
PORTER_NLTK = PorterStemmer(mode=PorterStemmer.NLTK_EXTENSIONS)

# check de como funcionan
print("-- Comparacion rapida --\n")
for palabra in ["news", "dying", "proceed", "innings", "connected"]:
    print(f"   {palabra:12} orig={PORTER_ORIG.stem(palabra):10} "
          f"nltk={PORTER_NLTK.stem(palabra)}")

-- Comparacion rapida --

   news         orig=new        nltk=news
   dying        orig=dy         nltk=die
   proceed      orig=proce      nltk=proceed
   innings      orig=in         nltk=inning
   connected    orig=connect    nltk=connect


In [64]:
def truncar(doc_txt, stemmer):
    """
    Trunca cada palabra del texto con el stemmer que se le pase

    Inputs:
    -------
    doc_txt: texto
    stemmer: algoritmo con el que se trunca

    Returns:
    -------
    str: El mismo texto con cada palabra reducida a su raiz

    """
    return " ".join(stemmer.stem(palabra) for palabra in doc_txt.split())

In [67]:
vc = 0
for doc in docs:
    doc["texto_porter_orig"] = truncar(doc["texto_nn"], PORTER_ORIG)
    doc["texto_porter_nltk"] = truncar(doc["texto_nn"], PORTER_NLTK)
    vc += 1

print(f"documentos procesados(trunc): {vc}")

print("\n\n--- SANITY CHECK ---\n")
print(f"id: {docs[0]['id']}")
print(f"fecha: {docs[0]['fecha']}")
print(f"pagina: {docs[0]['pagina']}")
print(f"largo: {len(docs[0]['texto'])} caracteres")
print(f"texto original: {len(docs[0]['texto'].split())} palabras")
print(f"texto sin nulos: {len(docs[0]['texto_nn'].split())} palabras")
print(f"porter 1980: {len(docs[0]['texto_porter_orig'].split())} palabras")
print(f"porter nltk: {len(docs[0]['texto_porter_nltk'].split())} palabras")
print()
print(f"texto original: {docs[0]['texto'][:300]} ...")
print(f"texto sin nulos: {docs[0]['texto_nn'][:300]} ...")
print(f"porter 1980: {docs[0]['texto_porter_orig'][:300]} ...")
print(f"porter nltk : {docs[0]['texto_porter_nltk'][:300]} ...")


documentos procesados(trunc): 423


--- SANITY CHECK ---

id: 017
fecha: 01/04/63
pagina: 020
largo: 5649 caracteres
texto original: 980 palabras
texto sin nulos: 445 palabras
porter 1980: 445 palabras
porter nltk: 445 palabras

texto original: THE ALLIES AFTER NASSAU IN DECEMBER 1960, THE U.S . FIRST PROPOSED TO HELP NATO DEVELOP ITS OWN NUCLEAR STRIKE FORCE . BUT EUROPE MADE NO ATTEMPT TO DEVISE A PLAN . LAST WEEK, AS THEY STUDIED THE NASSAU ACCORD BETWEEN PRESIDENT KENNEDY AND PRIME MINISTER MACMILLAN, EUROPEANS SAW EMERGING THE FIRST O ...
texto sin nulos: ALLIES NASSAU DECEMBER 1960 U.S PROPOSED HELP NATO DEVELOP NUCLEAR STRIKE FORCE EUROPE ATTEMPT DEVISE PLAN STUDIED NASSAU ACCORD PRESIDENT KENNEDY PRIME MINISTER MACMILLAN EUROPEANS SAW EMERGING OUTLINES NUCLEAR NATO U.S WANTS SUPPORT SPRANG ANGLO-U.S CRISIS CANCELLATION BUG-RIDDEN SKYBOLT MISSILE U ...
porter 1980: alli nassau decemb 1960 u. propos help nato develop nuclear strike forc europ attempt devis plan studi nassau accor

### En q difieren los dos modos

Sobre la coleccion completa, cuantos tokens truncan distinto y cuales
son los casos mas frecuentes

In [149]:
from collections import Counter

discrepancias = Counter()
tokens_totales = 0

for doc in docs:
    for palabra in doc["texto_nn"].split():
        tokens_totales += 1
        orig = PORTER_ORIG.stem(palabra)
        nltk_ = PORTER_NLTK.stem(palabra)
        if orig != nltk_:
            discrepancias[(palabra.lower(), orig, nltk_)] += 1

distintos = sum(discrepancias.values())

print(f"tokens revisados: {tokens_totales}")
print(f"truncan distinto: {distintos}")
print(f"porcentaje: {100 * distintos / tokens_totales:.2f}%")
print(f"formas distintas: {len(discrepancias)}")


def mas_frecuentes(llave, n=15):
    """
    Cuenta los terminos mas frecuentes de una variante del texto.
    Minusculiza para que crudo y sin vacias sean comparables con las
    variantes de Porter, que ya vienen en minusculas.

    Inputs:
    -------
    llave: Atributo del documento del que se cuenta
    n: Cuantos terminos devolver

    Returns:
    -------
    list: Pares termino y frecuencia, del mas comun al menos

    """
    cuenta = Counter()
    for doc in docs:
        cuenta.update(doc[llave].lower().split())
    return cuenta.most_common(n)


VARIANTES = [
    ("crudo", "texto"),
    ("sin vacias", "texto_nn"),
    ("porter 1980", "texto_porter_orig"),
    ("porter nltk", "texto_porter_nltk"),
]
ANCHO = 14

tops = [mas_frecuentes(llave) for _, llave in VARIANTES]

print("\n\n--- TOP 15 POR VARIANTE ---\n")
print("".join(f"{nombre:<{ANCHO + 8}}" for nombre, _ in VARIANTES))
print("-" * ((ANCHO + 8) * len(VARIANTES)))
for fila in range(15):
    print("".join(f"{t[fila][0]:<{ANCHO}}{t[fila][1]:>6}  " for t in tops))

print("\n\n--- PORTER TRUNC VS NLTK ---\n")
print(f"{'palabra':<16}{'orig 1980':<14}{'nltk':<14}veces")
for (palabra, orig, nltk_), n in discrepancias.most_common(15):
    print(f"{palabra:<16}{orig:<14}{nltk_:<14}{n}")



tokens revisados: 124030
truncan distinto: 2151
porcentaje: 1.73%
formas distintas: 434


--- TOP 15 POR VARIANTE ---

crudo                 sin vacias            porter 1980           porter nltk           
----------------------------------------------------------------------------------------
the            15809  u.s              726  u.               726  u.               726  
.              12148  government       583  govern           609  govern           609  
of              7219  new              549  new              603  new              549  
to              6303  party            378  parti            434  parti            434  
a               5781  de               335  nation           426  nation           426  
and             5501  minister         310  communist        420  communist        420  
in              5244  west             310  red              377  red              377  
"               4946  war              280  minist           370  minist        

## Extraer vocabulario
El bocabulario es como el universo de palabras de la coleccion... voy a comparar los diferentes vocabularios (de la coleccion) con el eliminado de palabras nulas y el trunc con porter solo para ver como se comportan

In [73]:
# Cuanto encoge el vocabulario cada modo
vocab_raw = {p.lower() for doc in docs for p in doc["texto"].split()}
vocab_nn = {p.lower() for doc in docs for p in doc["texto_nn"].split()}
vocab_orig = {p for doc in docs for p in doc["texto_porter_orig"].split()}
vocab_nltk = {p for doc in docs for p in doc["texto_porter_nltk"].split()}

print(f"vocabulario sin truncar: {len(vocab_raw)}")
print(f"vocabulario sin truncar (sin nulos): {len(vocab_nn)}")
print(f"con porter 1980: {len(vocab_orig)}")
print(f"con porter de nltk: {len(vocab_nltk)}")
print()
print(f"reduccion 1980: {100 * (1 - len(vocab_orig)/len(vocab_nn)):.4f}%")
print(f"reduccion nltk: {100 * (1 - len(vocab_nltk)/len(vocab_nn)):.4f}%")

vocabulario sin truncar: 29418
vocabulario sin truncar (sin nulos): 23338
con porter 1980: 16675
con porter de nltk: 16623

reduccion 1980: 28.5500%
reduccion nltk: 28.7728%


ahora el de cada documento

In [79]:
import pandas as pd


def vocabulario_por_documento(docs, key):
    """
    Arma la tabla de vocabulario con su frecuencia de termino

    Inputs:
    -------
    docs: Lista de documentos, cada uno un dict
    key: Atributo del que se cuentan los terminos

    Returns:
    -------
    DataFrame: Columnas doc_id, termino y tf

    """
    filas = [
        (doc["id"], termino, tf)
        for doc in docs
        for termino, tf in Counter(doc[key].split()).items()
    ]
    return pd.DataFrame(filas, columns=["doc_id", "termino", "tf"])

In [101]:
# Obtenemos los vocabulario por documento d las 4 variantes

# Texto crudo
vocab_raw = vocabulario_por_documento(docs, "texto")
por_doc_raw = vocab_raw.groupby("doc_id").agg(
    terminos=("termino", "size"),
    tokens=("tf", "sum"),
)
mat_raw = (vocab_raw.pivot(index="doc_id", columns="termino", values="tf")
          .fillna(0)
          .astype("int32"))

# Texto sin nulos
vocab_nn = vocabulario_por_documento(docs, "texto_nn")
por_doc_nn = vocab_nn.groupby("doc_id").agg(
    terminos=("termino", "size"),
    tokens=("tf", "sum"),
)
mat_nn = (vocab_nn.pivot(index="doc_id", columns="termino", values="tf")
          .fillna(0)
          .astype("int32"))

# Texto con trunc Porter original
vocab_porter_o = vocabulario_por_documento(docs, "texto_porter_orig")
por_doc_porter_o = vocab_porter_o.groupby("doc_id").agg(
    terminos=("termino", "size"),
    tokens=("tf", "sum"),
)
mat_porter_o = (vocab_porter_o.pivot(index="doc_id", columns="termino", values="tf")
          .fillna(0)
          .astype("int32"))

# Texto con trunc Porter nltk
vocab_nltk = vocabulario_por_documento(docs, "texto_porter_nltk")
por_doc_nltk = vocab_nltk.groupby("doc_id").agg(
    terminos=("termino", "size"),
    tokens=("tf", "sum"),
)
mat_nltk = (vocab_nltk.pivot(index="doc_id", columns="termino", values="tf")
          .fillna(0)
          .astype("int32"))


In [116]:

print("--- Datos del vocabulario raw ---")
print(f"pares documento-termino: {len(vocab_raw)}")
print(f"documentos             : {vocab_raw['doc_id'].nunique()}")
print(f"vocabulario global     : {vocab_raw['termino'].nunique()}")
print(f"tokens totales         : {vocab_raw['tf'].sum()}")

print("\n--- Matriz de terminos por documento (texto raw) ---\n")
display(mat_raw.iloc[:10, :12])

print("\n--- Matriz de terminos por documento (cortada a terminos que aparecen) ---\n")
top = vocab_raw.groupby("termino")["tf"].sum().nlargest(15).index
display(mat_raw.loc[mat_raw.index[:12], top])

--- Datos del vocabulario raw ---
pares documento-termino: 140756
documentos             : 423
vocabulario global     : 29418
tokens totales         : 263749

--- Matriz de terminos por documento (texto raw) ---



termino,!,"!960,","""","""AFRICAN","""AND","""APARTHEID","""BASED","""BUT","""DEMOCRATIC","""DO","""ENTANGLEMENT","""ENTRY"
doc_id,,,,,,,,,,,,
017,0,0,20,0,0,0,0,0,0,0,0,0
018,0,0,8,0,0,0,0,0,0,0,0,0
019,0,0,2,0,0,0,0,0,0,0,0,0
020,0,0,6,0,0,0,0,0,0,0,0,0
021,1,0,21,0,0,0,0,0,0,0,0,0
023,0,0,7,0,0,0,0,0,0,0,0,0
024,0,0,10,0,0,0,0,0,0,0,0,0
025,0,0,2,0,0,0,0,0,0,0,0,0
026,0,0,21,0,0,0,0,0,0,0,0,0



--- Matriz de terminos por documento (cortada a terminos que aparecen) ---



termino,THE,.,OF,TO,A,AND,IN,"""",THAT,FOR,WAS,WITH,HIS,IS,HE
doc_id,,,,,,,,,,,,,,,
017,52,52,16,27,17,16,10,20,25,8,4,8,6,6,2
018,12,12,7,5,4,3,5,8,2,5,6,1,4,0,1
019,46,28,16,18,11,12,7,2,4,6,7,1,1,0,1
020,9,10,3,6,5,4,2,6,0,1,3,1,1,1,2
021,51,57,18,12,21,17,21,21,8,2,7,7,5,3,6
023,30,29,20,14,15,9,12,7,6,3,4,2,9,3,6
024,55,42,39,27,10,15,16,10,12,12,7,6,2,4,1
025,7,6,2,1,5,5,1,2,3,1,2,3,0,1,0
026,38,19,20,7,6,10,10,21,3,3,4,7,3,3,3


In [ ]:
print("--- Datos del vocabulario nn ---")
print(f"pares documento-termino: {len(vocab_nn)}")
print(f"documentos             : {vocab_nn['doc_id'].nunique()}")
print(f"vocabulario global     : {vocab_nn['termino'].nunique()}")
print(f"tokens totales         : {vocab_nn['tf'].sum()}")

print("\n--- Matriz de terminos por documento (texto sin neutros) ---\n")
display(mat_nn.iloc[:10, :12])

print("\n--- Matriz de terminos por documento (cortada a terminos que aparecen) ---\n")
top = vocab_nn.groupby("termino")["tf"].sum().nlargest(15).index
display(mat_nn.loc[mat_nn.index[:12], top])


--- Datos del vocabulario nn ---
pares documento-termino: 96865
documentos             : 423
vocabulario global     : 23338
tokens totales         : 124030

--- Matriz de terminos por documento (texto sin neutros) ---



termino,00,1,"1,000","1,000,000","1,000-YEAR-OLD","1,000TH","1,000TO-1","1,014","1,100","1,100,000","1,193","1,200"
doc_id,,,,,,,,,,,,
017,0,1,0,0,0,0,0,0,0,0,0,0
018,0,0,0,0,0,0,0,0,0,0,0,0
019,0,0,0,0,0,0,0,0,0,0,0,0
020,0,0,0,0,0,0,0,0,0,0,0,0
021,0,0,0,0,0,0,0,0,0,0,0,0
023,0,0,0,0,0,0,0,0,0,0,0,0
024,0,0,0,0,0,0,0,0,0,0,0,0
025,0,0,0,0,0,0,0,0,0,0,0,0
026,0,0,0,0,0,0,0,0,0,0,0,0



--- Matriz de terminos por documento (cortada a terminos que aparecen) ---



termino,U.S,GOVERNMENT,NEW,PARTY,DE,MINISTER,WEST,SOUTH,WAR,COMMUNIST,RED,VIET,BRITISH,PRESIDENT,WORLD
doc_id,,,,,,,,,,,,,,,
017,15,2,0,0,11,2,1,0,0,0,0,0,4,2,0
018,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0
019,2,0,0,0,0,0,3,0,0,3,0,0,0,0,0
020,3,0,0,0,0,0,0,0,0,0,1,0,0,0,0
021,2,1,0,0,0,0,0,0,1,0,1,0,2,0,0
023,2,0,1,1,0,0,1,0,0,0,0,0,0,2,0
024,1,1,0,0,0,1,0,0,1,0,2,0,1,0,0
025,0,0,0,0,0,0,0,0,0,0,4,0,0,0,0
026,1,0,1,1,0,1,1,0,0,0,0,0,2,0,0


In [117]:
print("--- Datos del vocabulario porter_orig ---")
print(f"pares documento-termino: {len(vocab_porter_o)}")
print(f"documentos             : {vocab_porter_o['doc_id'].nunique()}")
print(f"vocabulario global     : {vocab_porter_o['termino'].nunique()}")
print(f"tokens totales         : {vocab_porter_o['tf'].sum()}")

print("\n--- Matriz de terminos por documento (texto porter_orig) ---\n")
display(mat_porter_o.iloc[:10, :12])

print("\n--- Matriz de terminos por documento (cortada a terminos que aparecen) ---\n")
top = vocab_porter_o.groupby("termino")["tf"].sum().nlargest(15).index
display(mat_porter_o.loc[mat_porter_o.index[:12], top])



--- Datos del vocabulario porter_orig ---
pares documento-termino: 92091
documentos             : 423
vocabulario global     : 16675
tokens totales         : 124030

--- Matriz de terminos por documento (texto porter_orig) ---



termino,00,1,"1,000","1,000,000","1,000-year-old","1,000th","1,000to-1","1,014","1,100","1,100,000","1,193","1,200"
doc_id,,,,,,,,,,,,
017,0,1,0,0,0,0,0,0,0,0,0,0
018,0,0,0,0,0,0,0,0,0,0,0,0
019,0,0,0,0,0,0,0,0,0,0,0,0
020,0,0,0,0,0,0,0,0,0,0,0,0
021,0,0,0,0,0,0,0,0,0,0,0,0
023,0,0,0,0,0,0,0,0,0,0,0,0
024,0,0,0,0,0,0,0,0,0,0,0,0
025,0,0,0,0,0,0,0,0,0,0,0,0
026,0,0,0,0,0,0,0,0,0,0,0,0



--- Matriz de terminos por documento (cortada a terminos que aparecen) ---



termino,u.,govern,new,parti,nation,communist,red,minist,de,forc,west,war,polit,south,countri
doc_id,,,,,,,,,,,,,,,
017,15,2,0,0,4,0,0,2,11,10,1,0,0,0,1
018,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0
019,2,0,0,0,0,4,0,0,0,0,3,0,0,0,0
020,3,0,0,0,0,0,1,0,0,0,0,0,0,0,0
021,2,1,0,0,0,0,1,0,0,5,0,1,1,0,0
023,2,0,1,1,1,0,0,0,0,0,1,0,1,0,2
024,1,1,0,0,3,0,2,1,0,0,0,1,0,0,1
025,0,0,0,0,0,0,4,0,0,0,0,0,0,0,1
026,1,1,2,1,0,0,0,1,0,0,1,0,0,0,0


In [118]:
print("--- Datos del vocabulario porter_nltk ---")
print(f"pares documento-termino: {len(vocab_nltk)}")
print(f"documentos             : {vocab_nltk['doc_id'].nunique()}")
print(f"vocabulario global     : {vocab_nltk['termino'].nunique()}")
print(f"tokens totales         : {vocab_nltk['tf'].sum()}")

print("\n--- Matriz de terminos por documento (texto porter_nltk) ---\n")
display(mat_nltk.iloc[:10, :12])

print("\n--- Matriz de terminos por documento (cortada a terminos que aparecen) ---\n")
top = vocab_nltk.groupby("termino")["tf"].sum().nlargest(15).index
display(mat_nltk.loc[mat_nltk.index[:12], top])

--- Datos del vocabulario porter_nltk ---
pares documento-termino: 92040
documentos             : 423
vocabulario global     : 16623
tokens totales         : 124030

--- Matriz de terminos por documento (texto porter_nltk) ---



termino,00,1,"1,000","1,000,000","1,000-year-old","1,000th","1,000to-1","1,014","1,100","1,100,000","1,193","1,200"
doc_id,,,,,,,,,,,,
017,0,1,0,0,0,0,0,0,0,0,0,0
018,0,0,0,0,0,0,0,0,0,0,0,0
019,0,0,0,0,0,0,0,0,0,0,0,0
020,0,0,0,0,0,0,0,0,0,0,0,0
021,0,0,0,0,0,0,0,0,0,0,0,0
023,0,0,0,0,0,0,0,0,0,0,0,0
024,0,0,0,0,0,0,0,0,0,0,0,0
025,0,0,0,0,0,0,0,0,0,0,0,0
026,0,0,0,0,0,0,0,0,0,0,0,0



--- Matriz de terminos por documento (cortada a terminos que aparecen) ---



termino,u.,govern,new,parti,nation,communist,red,minist,de,forc,west,war,polit,south,countri
doc_id,,,,,,,,,,,,,,,
017,15,2,0,0,4,0,0,2,11,10,1,0,0,0,1
018,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0
019,2,0,0,0,0,4,0,0,0,0,3,0,0,0,0
020,3,0,0,0,0,0,1,0,0,0,0,0,0,0,0
021,2,1,0,0,0,0,1,0,0,5,0,1,1,0,0
023,2,0,1,1,1,0,0,0,0,0,1,0,1,0,2
024,1,1,0,0,3,0,2,1,0,0,0,1,0,0,1
025,0,0,0,0,0,0,4,0,0,0,0,0,0,0,1
026,1,1,1,1,0,0,0,1,0,0,1,0,0,0,0


## Queries
Ahora procesmaos la queries y sacamos sus vectores

In [121]:
RUTA_COLECCION = Path("../time")
ARCHIVO_QUERIES = RUTA_COLECCION / "TIME.QUE"

MARCA_QUERY = "*FIND"

In [122]:
def leer_queries(ruta):
    """
    Separa TIME.QUE en sus consultas, reusando logica de docs

    Inputs:
    -------
    ruta: path al archivo TIME.QUE

    Returns:
    -------
    list: Un dict por consulta, con id y texto

    """
    queries, actual, cuerpo = [], None, []

    for line in ruta.read_text(encoding="ascii").splitlines():
        line = line.strip()

        if line.startswith(MARCA_QUERY):
            if actual is not None:
                actual["texto"] = " ".join(cuerpo)
                queries.append(actual)
            actual = {"id": line.split()[1]}
            cuerpo = []
        elif line == MARCA_FIN:
            break
        elif line:
            cuerpo.append(line)

    if actual is not None:
        actual["texto"] = " ".join(cuerpo)
        queries.append(actual)

    return queries


In [ ]:
queries = leer_queries(ARCHIVO_QUERIES)

for q in queries:
    q["texto_nn"] = eliminar_palabras(q["texto"])
    q["texto_porter_orig"] = truncar(q["texto_nn"], PORTER_ORIG)
    q["texto_porter_nltk"] = truncar(q["texto_nn"], PORTER_NLTK)


In [128]:
print(f"queries ({len(queries)})")

print("\n\n--- SANITY CHECK ---\n")
for vc in range(0, min(5, len(queries))):
    print(f"\n--- Query {vc+1} ---")
    print(f"id: {queries[vc]['id']}")
    print(f"largo: {len(queries[vc]['texto'])} caracteres")
    print(f"query original: {len(queries[vc]['texto'].split())} palabras")
    print(f"query sin nulos: {len(queries[vc]['texto_nn'].split())} palabras")
    print(f"query 1980: {len(queries[vc]['texto_porter_orig'].split())} palabras")
    print(f"query nltk: {len(queries[vc]['texto_porter_nltk'].split())} palabras")
    print()
    print(f"texto original: {queries[vc]['texto']}")
    print(f"texto sin nulos: {queries[vc]['texto_nn']}")
    print(f"query 1980: {queries[vc]['texto_porter_orig']}")
    print(f"query nltk: {queries[vc]['texto_porter_nltk']}")

queries (83)


--- SANITY CHECK ---


--- Query 1 ---
id: 1
largo: 84 caracteres
query original: 13 palabras
query sin nulos: 9 palabras
query 1980: 9 palabras
query nltk: 9 palabras

texto original: KENNEDY ADMINISTRATION PRESSURE ON NGO DINH DIEM TO STOP SUPPRESSING THE BUDDHISTS .
texto sin nulos: KENNEDY ADMINISTRATION PRESSURE NGO DINH DIEM STOP SUPPRESSING BUDDHISTS
query 1980: kennedi administr pressur ngo dinh diem stop suppress buddhist
query nltk: kennedi administr pressur ngo dinh diem stop suppress buddhist

--- Query 2 ---
id: 2
largo: 121 caracteres
query original: 20 palabras
query sin nulos: 13 palabras
query 1980: 13 palabras
query nltk: 13 palabras

texto original: EFFORTS OF AMBASSADOR HENRY CABOT LODGE TO GET VIET NAM'S PRESIDENT DIEM TO CHANGE HIS POLICIES OF POLITICAL REPRESSION .
texto sin nulos: EFFORTS AMBASSADOR HENRY CABOT LODGE VIET NAM'S PRESIDENT DIEM CHANGE POLICIES POLITICAL REPRESSION
query 1980: effort ambassador henri cabot lodg viet nam' presid diem 

In [ ]:
vocab_queries_raw = vocabulario_por_documento(queries, "texto")

# RAW
mat_queries_raw = (vocab_queries_raw
               .pivot(index="doc_id", columns="termino", values="tf")
               .reindex(columns=mat_raw.columns, fill_value=0)
               .fillna(0)
               .astype("int32")
               .rename_axis(index="query_id"))

mat_queries_raw = mat_queries_raw.reindex(sorted(mat_queries_raw.index, key=int))

vocab_queries_nn = vocabulario_por_documento(queries, "texto_nn")

# No nulos
mat_queries_nn = (vocab_queries_nn
               .pivot(index="doc_id", columns="termino", values="tf")
               .reindex(columns=mat_nn.columns, fill_value=0)
               .fillna(0)
               .astype("int32")
               .rename_axis(index="query_id"))

mat_queries_nn = mat_queries_nn.reindex(sorted(mat_queries_nn.index, key=int))

# porter 
vocab_queries_porter = vocabulario_por_documento(queries, "texto_porter_orig")

mat_queries_porter = (vocab_queries_porter
               .pivot(index="doc_id", columns="termino", values="tf")
               .reindex(columns=mat_porter_o.columns, fill_value=0)
               .fillna(0)
               .astype("int32")
               .rename_axis(index="query_id"))

mat_queries_porter = mat_queries_porter.reindex(sorted(mat_queries_porter.index, key=int))

# NLTK
vocab_queries_nltk = vocabulario_por_documento(queries, "texto_porter_nltk")

mat_queries_nltk = (vocab_queries_nltk
               .pivot(index="doc_id", columns="termino", values="tf")
               .reindex(columns=mat_nltk.columns, fill_value=0)
               .fillna(0)
               .astype("int32")
               .rename_axis(index="query_id"))

mat_queries_nltk = mat_queries_nltk.reindex(sorted(mat_queries_nltk.index, key=int))



In [ ]:
print("--- Datos del vocabulario queries raw ---")
print(f"pares documento-termino: {len(vocab_queries_raw)}")
print(f"documentos             : {vocab_queries_raw['doc_id'].nunique()}")
print(f"vocabulario global     : {vocab_queries_raw['termino'].nunique()}")
print(f"tokens totales         : {vocab_queries_raw['tf'].sum()}")

print("\n--- Matriz de terminos por documento (texto raw) ---\n")
display(mat_queries_raw.iloc[:10, :12])

print("\n--- Matriz de terminos por documento (cortada a terminos que aparecen) ---\n")
top = vocab_queries_raw.groupby("termino")["tf"].sum().nlargest(15).index
display(mat_queries_raw.loc[mat_queries_raw.index[:12], top])



--- Datos del vocabulario queries raw ---
pares documento-termino: 1228
documentos             : 83
vocabulario global     : 552
tokens totales         : 1341

--- Matriz de terminos por documento (texto raw) ---



termino,!,"!960,","""","""AFRICAN","""AND","""APARTHEID","""BASED","""BUT","""DEMOCRATIC","""DO","""ENTANGLEMENT","""ENTRY"
query_id,,,,,,,,,,,,
1,0,0,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,0,0
5,0,0,0,0,0,0,0,0,0,0,0,0
6,0,0,0,0,0,0,0,0,0,0,0,0
7,0,0,0,0,0,0,0,0,0,0,0,0
8,0,0,0,0,0,0,0,0,0,0,0,0
9,0,0,0,0,0,0,0,0,0,0,0,0



--- Matriz de terminos por documento (cortada a terminos que aparecen) ---



termino,.,THE,OF,IN,TO,AND,BY,ON,NATIONS,A,FOR,UNITED,WHICH,ITS,PRESIDENT
query_id,,,,,,,,,,,,,,,
1,1,1,0,0,1,0,0,1,0,0,0,0,0,0,0
2,1,0,2,0,2,0,0,0,0,0,0,0,0,0,1
3,1,2,2,2,0,0,0,0,0,0,0,1,0,0,0
4,2,1,0,1,0,0,0,0,0,0,0,0,1,0,1
5,1,1,0,1,0,0,0,0,0,0,0,0,0,0,0
6,1,0,0,1,1,1,2,0,0,0,0,0,0,0,0
7,2,0,1,0,1,0,1,0,0,0,0,0,0,0,0
8,2,2,3,1,0,1,0,1,0,0,0,0,0,0,0
9,1,1,1,0,1,0,0,0,0,0,0,0,0,0,0


In [142]:
print("--- Datos del vocabulario queries nn ---")
print(f"pares documento-termino: {len(vocab_queries_nn)}")
print(f"documentos             : {vocab_queries_nn['doc_id'].nunique()}")
print(f"vocabulario global     : {vocab_queries_nn['termino'].nunique()}")
print(f"tokens totales         : {vocab_queries_nn['tf'].sum()}")

print("\n--- Matriz de terminos por documento (texto nn) ---\n")
display(mat_queries_nn.iloc[:10, :12])

print("\n--- Matriz de terminos por documento (cortada a terminos que aparecen) ---\n")
top = vocab_queries_nn.groupby("termino")["tf"].sum().nlargest(15).index
display(mat_queries_nn.loc[mat_queries_nn.index[:12], top])

--- Datos del vocabulario queries nn ---
pares documento-termino: 697
documentos             : 83
vocabulario global     : 449
tokens totales         : 722

--- Matriz de terminos por documento (texto nn) ---



termino,00,1,"1,000","1,000,000","1,000-YEAR-OLD","1,000TH","1,000TO-1","1,014","1,100","1,100,000","1,193","1,200"
query_id,,,,,,,,,,,,
1,0,0,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,0,0
5,0,0,0,0,0,0,0,0,0,0,0,0
6,0,0,0,0,0,0,0,0,0,0,0,0
7,0,0,0,0,0,0,0,0,0,0,0,0
8,0,0,0,0,0,0,0,0,0,0,0,0
9,0,0,0,0,0,0,0,0,0,0,0,0



--- Matriz de terminos por documento (cortada a terminos que aparecen) ---



termino,NATIONS,UNITED,PRESIDENT,PREMIER,U.S,AFRICAN,COMMUNIST,NEW,PARTY,SOUTH,SOVIET,COUNTRIES,FEDERATION,KHRUSHCHEV,MALAYSIA
query_id,,,,,,,,,,,,,,,
1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0
3,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0
4,0,0,1,0,1,0,0,1,0,1,0,0,0,0,0
5,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
6,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0
7,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
8,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1
9,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1


In [143]:
print("--- Datos del vocabulario queries porter ---")
print(f"pares documento-termino: {len(vocab_queries_porter)}")
print(f"documentos             : {vocab_queries_porter['doc_id'].nunique()}")
print(f"vocabulario global     : {vocab_queries_porter['termino'].nunique()}")
print(f"tokens totales         : {vocab_queries_porter['tf'].sum()}")

print("\n--- Matriz de terminos por documento (texto porter) ---\n")
display(mat_queries_porter.iloc[:10, :12])

print("\n--- Matriz de terminos por documento (cortada a terminos que aparecen) ---\n")
top = vocab_queries_porter.groupby("termino")["tf"].sum().nlargest(15).index
display(mat_queries_porter.loc[mat_queries_porter.index[:12], top])

--- Datos del vocabulario queries porter ---
pares documento-termino: 695
documentos             : 83
vocabulario global     : 413
tokens totales         : 722

--- Matriz de terminos por documento (texto porter) ---



termino,00,1,"1,000","1,000,000","1,000-year-old","1,000th","1,000to-1","1,014","1,100","1,100,000","1,193","1,200"
query_id,,,,,,,,,,,,
1,0,0,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,0,0
5,0,0,0,0,0,0,0,0,0,0,0,0
6,0,0,0,0,0,0,0,0,0,0,0,0
7,0,0,0,0,0,0,0,0,0,0,0,0
8,0,0,0,0,0,0,0,0,0,0,0,0
9,0,0,0,0,0,0,0,0,0,0,0,0



--- Matriz de terminos por documento (cortada a terminos que aparecen) ---



termino,nation,unit,presid,premier,propos,u.,parti,state,african,communist,countri,forc,involv,new,south
query_id,,,,,,,,,,,,,,,
1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0
3,0,1,0,0,0,0,0,1,0,0,0,0,0,0,1
4,0,0,1,0,0,1,0,0,0,0,0,0,0,1,1
5,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0
6,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
7,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0
8,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
9,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [144]:
print("--- Datos del vocabulario queries nltk ---")
print(f"pares documento-termino: {len(vocab_queries_nltk)}")
print(f"documentos             : {vocab_queries_nltk['doc_id'].nunique()}")
print(f"vocabulario global     : {vocab_queries_nltk['termino'].nunique()}")
print(f"tokens totales         : {vocab_queries_nltk['tf'].sum()}")

print("\n--- Matriz de terminos por documento (texto nltk) ---\n")
display(mat_queries_nltk.iloc[:10, :12])

print("\n--- Matriz de terminos por documento (cortada a terminos que aparecen) ---\n")
top = vocab_queries_nltk.groupby("termino")["tf"].sum().nlargest(15).index
display(mat_queries_nltk.loc[mat_queries_nltk.index[:12], top])

--- Datos del vocabulario queries nltk ---
pares documento-termino: 695
documentos             : 83
vocabulario global     : 413
tokens totales         : 722

--- Matriz de terminos por documento (texto nltk) ---



termino,00,1,"1,000","1,000,000","1,000-year-old","1,000th","1,000to-1","1,014","1,100","1,100,000","1,193","1,200"
query_id,,,,,,,,,,,,
1,0,0,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,0,0
5,0,0,0,0,0,0,0,0,0,0,0,0
6,0,0,0,0,0,0,0,0,0,0,0,0
7,0,0,0,0,0,0,0,0,0,0,0,0
8,0,0,0,0,0,0,0,0,0,0,0,0
9,0,0,0,0,0,0,0,0,0,0,0,0



--- Matriz de terminos por documento (cortada a terminos que aparecen) ---



termino,nation,unit,presid,premier,propos,u.,parti,state,african,communist,countri,forc,involv,new,south
query_id,,,,,,,,,,,,,,,
1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0
3,0,1,0,0,0,0,0,1,0,0,0,0,0,0,1
4,0,0,1,0,0,1,0,0,0,0,0,0,0,1,1
5,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0
6,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
7,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0
8,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
9,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [145]:
def export_voc(vocab, ruta, prefijo="Doc", orden_numerico=False):
    """
    Escribe el vocabulario en el formato que pide el enunciado

    Inputs:
    -------
    vocab: DataFrame con columnas doc_id, termino y tf
    ruta: Archivo de salida
    prefijo: Lo que va antes del id, Doc o Query
    orden_numerico: True cuando los ids no traen ceros a la izquierda,
                    para que 10 no quede antes que 2

    Returns:
    -------
    int: Numero de lineas escritas

    """
    ids = vocab["doc_id"].unique()
    if orden_numerico:
        ids = sorted(ids, key=int)

    grupos = dict(tuple(vocab.groupby("doc_id", sort=False)))

    with open(ruta, "w", encoding="utf-8") as salida:
        for id_ in ids:
            grupo = grupos[id_].sort_values(
                ["tf", "termino"], ascending=[False, True])
            pares = " ".join(f"{t} {n}" for t, n
                             in zip(grupo["termino"], grupo["tf"]))
            salida.write(f"{prefijo}{id_} {pares}\n")

    return len(ids)


In [146]:
salidas = [
    (vocab_raw,     "vocab_raw.txt",     "Doc",   False),
    (vocab_nn,      "vocab_nn.txt",      "Doc",   False),
    (vocab_porter_o,  "vocab_porter.txt",  "Doc",   False),
    (vocab_queries_raw, "vocab_queries_raw.txt", "Query", True),
    (vocab_queries_nn, "vocab_queries_nn.txt", "Query", True),
    (vocab_queries_porter, "vocab_queries_porter.txt", "Query", True),
    (vocab_queries_nltk, "vocab_queries_nltk.txt", "Query", True),
]

for vocab, nombre, prefijo, num in salidas:
    n = export_voc(vocab, nombre, prefijo, num)
    print(f"{nombre:22} {n:4} lineas")


vocab_raw.txt           423 lineas
vocab_nn.txt            423 lineas
vocab_porter.txt        423 lineas
vocab_queries_raw.txt    83 lineas
vocab_queries_nn.txt     83 lineas
vocab_queries_porter.txt   83 lineas
vocab_queries_nltk.txt   83 lineas
